In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!pip install -q google-genai

In [ ]:
import re
import copy

import pandas as pd
import google.generativeai as genai
from google.genai.types import HarmCategory, HarmBlockThreshold
import time
import os

import json
import copy
from tqdm import tqdm

import glob

In [ ]:
SELECTED_MODEL = 'gemini-2.5-pro-preview-03-25'
max_output_tokens =  65536

training_df = pd.read_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/01_data_preparation/train.csv')
batch_size = 30

In [ ]:
def create_model():
    with open('/content/drive/MyDrive/Project/secrets/Gemini-API-Key.txt', 'r') as f:
        GEMINI_API_KEY = f.read().strip()

    safety_settings_options = {
        HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
        HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_NONE,
    }

    genai.configure(api_key=GEMINI_API_KEY)

    generation_config = {
        "temperature": 1,  # Controls randomness (0 = deterministic, 1 = highly random)
        "top_p": 0.95,     # Higher values make the model more creative
        "max_output_tokens": max_output_tokens,
        "response_mime_type": "text/plain",
    }

    model = genai.GenerativeModel(
        model_name=SELECTED_MODEL,
        generation_config=generation_config,
        safety_settings=safety_settings_options,
    )
    print(model)
    return model

model =     create_model()

genai.GenerativeModel(
    model_name='models/gemini-2.5-pro-preview-03-25',
    generation_config={'temperature': 1, 'top_p': 0.95, 'max_output_tokens': 65536, 'response_mime_type': 'text/plain'},
    safety_settings={<HarmCategory.HARM_CATEGORY_HARASSMENT: 7>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_HATE_SPEECH: 8>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 9>: <HarmBlockThreshold.BLOCK_NONE: 4>, <HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 10>: <HarmBlockThreshold.BLOCK_NONE: 4>},
    tools=None,
    system_instruction=None,
    cached_content=None
)


In [ ]:
prompt = f'''
I will give you Arabic sarcastic tweets. For each tweet, analyize in details the following:
- What is the topic of the tweet?
- What is the context of the tweet?
- What is the dialect of the tweet?
Dialect can be one of the following:
* msa: Modern Standard Arabic (الفصحى) used in formal writing and media,
* egypt: Egyptian dialect (العامية المصرية) used in Egypt and Sudan,
* levant: Levantine dialect (اللهجة الشامية) used in Palestine, Jordan, Syria and Lebanon,
* gulf: Gulf dialect (اللهجة الخليجية) used in Saudi Arabia, UAE, Qatar, Bahrain, Yemen, Oman, Iraq and Kuwait,
* magreb: Maghrebi dialect (اللهجة المغاربية) used in North African Arab countries including Algeria, Libya, Tunisia and Morocco
- In which year was this tweet written based on the topic/context of it.
- In which country was that tweet written.
- Why tweet is sarcastic?
- What is the sentiment polarity while ignoring being a sarcastic or non-sarcastic tweet? polarity can be one of the following neutral sentiment, negative sentiment, positive sentiment.
- What is the type of sarcasm used?
- Does it contain impolite words?
- Does it contain hateful words?
- Does the tweet contain contradictory statements?
- Does the tweet have hidden meanings?
- Does the tweet contain double entendre?
- Does the tweet use metaphors to compare things to other things?
- Is it a long or a short tweet? why this is the best lengh of the tweet?
- How many arabic words used?
- Analyize the personality and the circumstances of the person who wrote the tweet.
- What motives led this person to say this tweet?
- What is the relationship between dialect and country and year and context of tweet and sarcasm used and personality and the circumstances?

Important: Never qoute any words of the used words in the tweets or its English translatation in the response.
Analysis must never mention any of the used words or the translation of the used words.
Keep the same exact order of the analysis as the passed tweets. Each analysis must be in exactelly 1 line. Do not include in response any explanation or further strings.
Number of response lines must = number of tweets with format tweet_content:analysis

Below is an example of analysis. Learn from this example how to analize, and how tweet_content:analysis response format must be for each tweet\n
عبدالله العذبه يسال كيف تتخندقون مع المعتدي الروسي يا اخوان ام ان دماء السوريين تهون لاجل عيوناردوغان والحزب:**Topic:** The tweet discusses political alignment concerning the Syrian conflict, specifically questioning the rationale behind supporting or siding with Russia, given its role in the conflict, and suggesting this alignment is driven by allegiance to specific political figures or parties rather than moral considerations for the victims. **Context:** The context is the ongoing Syrian civil war and the complex web of international and regional alliances. It specifically addresses factions or groups within the Arab world who are perceived as aligning with Russia, despite Russia's military actions in Syria. This alignment is contrasted with the suffering of Syrians and linked to political relationships with Turkey's leader and an associated political entity. The tweet likely originates from a period of heightened debate about foreign intervention in Syria and regional political polarization. The mention of a specific media personality suggests it's part of a broader media or public discourse on these geopolitical issues. **Dialect:** The language used is predominantly **msa** (Modern Standard Arabic). The vocabulary and sentence structure are formal and align with the language typically used in news commentary and political discourse across the Arab world. **Year:** Based on the context – Russia's established role as an ""aggressor"" in Syria (post-2015 intervention), the prominence of the named Turkish leader in regional politics, and the critical tone often associated with specific regional media outlets (linked to the named individual) during periods of intense geopolitical rivalry – the tweet was likely written between **2017 and 2020**. This period saw significant discussion about alignments related to the Syrian conflict amidst broader regional tensions. **Country:** The named individual in the tweet is strongly associated with media outlets based in **Qatar**. The political viewpoint expressed—criticizing alignment with Russia regarding Syria and framing it in relation to the Turkish leader—aligns well with narratives prominent in Qatari media during the aforementioned period, especially when criticizing regional rivals. **Why Sarcastic:** The sarcasm stems from the rhetorical nature of the question posed. It's not a genuine inquiry but an accusation framed as a question. It implies that the only conceivable reason for the questioned alignment is a morally reprehensible disregard for human suffering (""Syrian blood"") in favor of political expediency related to the named leader and party. The contrast between the expected moral stance and the alleged political motivation creates the sarcastic effect, mocking the priorities of the group being addressed. **Type of Sarcasm:** This is primarily **rhetorical sarcasm** blended with **indignant sarcasm**. The question is asked to highlight the perceived absurdity and moral failing of the action, not to elicit an actual answer. It conveys strong disapproval and mocks the stated or implied justifications for the political alignment. **Impolite Words:** The tweet does not contain overtly vulgar or taboo impolite words. The language remains within the bounds of formal political criticism, although the tone is harsh and accusatory. **Hateful Words:** While highly critical and employing strong, emotive language regarding the value of human life, it doesn't clearly constitute hate speech aimed at inciting violence or discrimination against a protected group based on identity. It's harsh political condemnation targeting a political stance and its perceived motivations. **Contradictory Statements:** The tweet itself does not contain internal contradictions. It presents a consistent critical argument – that aligning with the ""aggressor"" contradicts any claim of concern for Syrian victims and implies hypocrisy. **Hidden Meanings:** While the main point is quite direct, the ""hidden meaning"" lies in the unstated but strongly implied accusation of moral bankruptcy and political subservience directed at the group being questioned. The specific identity of ""the party"" might also carry specific connotations depending on the contemporary political context understood by the audience. **Double Entendre:** The tweet does not appear to use double entendre. The language is politically direct and focused on a specific critique. **Metaphors:** Yes, the tweet uses metaphors. The concept of ""taking a trench"" is used metaphorically for aligning strongly with a side. The idea of blood being ""cheap"" or disregarded is a metaphor for devaluing human lives. The phrase indicating doing something ""for the sake of"" or to please someone is also a common metaphorical idiom. **Length:** It is a **short** tweet. This length is optimal for Twitter as it allows for quick consumption and high impact. Sarcastic points, especially those using rhetorical questions or sharp contrasts, are often most effective when concise. **Arabic Words Used:** There are 20 Arabic words in the core message of the tweet (excluding the RT markers and URL). **Non-Arabic Words Used:** One proper name of non-Arabic origin (Turkish) is used, transliterated into Arabic script. **Hashtags/URLs/Special Characters:** There is one truncated URL and elements like `RT` and `PEP 144` which seem to be retweet information or tags rather than standard hashtags (`#`). So, 3 such items. **Personality/Circumstances:** The author (or the person being retweeted) appears to be politically informed, highly opinionated, and deeply invested in the geopolitical conflicts of the region, particularly the Syrian war. They likely hold strong views against certain political actors and alliances. They seem morally outraged (or portray such outrage) by perceived hypocrisy. Their circumstances likely involve being active in or closely following political media discourse, using social media as a platform for expressing sharp criticism and shaping narratives, potentially aligned with a specific political camp (likely Qatari/Turkish axis in this context). **Motives:** The primary motives appear to be: a) criticizing and delegitimizing the political actions and alliances of opponents; b) expressing solidarity with the Syrian victims (or using their plight for political point-scoring); c) reinforcing a particular political narrative that portrays rivals as morally compromised; d) potentially swaying public opinion among followers. **Relationship:** The use of **MSA (dialect)** suits the formal **political topic** and allows for broad reach, fitting the context of media-related commentary likely originating from **Qatar (country)** around **2017-2020 (year)** when **regional tensions regarding Syria, Russia, and Turkey (context)** were high. The **rhetorical/indignant sarcasm** is a tool used by the **politically engaged/critical author (personality)** to sharply critique perceived hypocrisy regarding the **Syrian conflict**. This specific type of **sarcasm**, delivered concisely in MSA, aims to delegitimize opponents' stances within the heated **political circumstances** of the time, driven by **motives** of political opposition and narrative shaping. The elements are interconnected, painting a picture of pointed political commentary within a specific regional conflict dynamic.
Response must never contain any empty or new lines like \n or \r
'''

chat_sarcastic_analysis = model.start_chat(history=[])
chat_sarcastic_analysis.send_message(prompt)

#######################
#######################


prompt = f'''
I will give you Arabic tweets. For each tweet, analyize in details the following:
- What is the topic of the tweet?
- What is the context of the tweet?
- What is the dialect of the tweet?
Dialect can be one of the following:
* msa: Modern Standard Arabic (الفصحى) used in formal writing and media,
* egypt: Egyptian dialect (العامية المصرية) used in Egypt and Sudan,
* levant: Levantine dialect (اللهجة الشامية) used in Palestine, Jordan, Syria and Lebanon,
* gulf: Gulf dialect (اللهجة الخليجية) used in Saudi Arabia, UAE, Qatar, Bahrain, Yemen, Oman, Iraq and Kuwait,
* magreb: Maghrebi dialect (اللهجة المغاربية) used in North African Arab countries including Algeria, Libya, Tunisia and Morocco
- In which year was this tweet written based on the topic/context of it.
- In which country was that tweet written.
- What is the sentiment polarity while ignoring being a sarcastic or non-sarcastic tweet? polarity can be one of the following neutral sentiment, negative sentiment, positive sentiment.
- Does it contain impolite words?
- Does it contain hateful words?
- Does the tweet contain contradictory statements?
- Does the tweet have hidden meanings?
- Does the tweet contain double entendre?
- Does the tweet use metaphors to compare things to other things?
- Is it a long or a short tweet? why this is the best lengh of the tweet?
- How many arabic words used?
- Analyize the personality and the circumstances of the person who wrote the tweet.
- What motives led this person to say this tweet?
- What is the relationship between dialect and country and year and context of tweet and personality and the circumstances?

Important: Never qoute any words of the used words in the tweets or its English translatation in the response.
Analysis must never mention any of the used words or the translation of the used words.
Keep the same exact order of the analysis as the passed tweets. Each analysis must be in exactelly 1 line. Do not include in response any explanation or further strings.
Number of response lines must = number of tweets with format tweet_content:analysis

Below is an example of analysis. Learn from this example how to analize, and how tweet_content:analysis response format must be for each tweet\n
هؤلاء صناعه المخابرات الامريكيه كما ذكرت هيلاري كلنتون وداعش ايرانيه:**Topic:** The origin and alleged creators/backers of specific groups involved in regional conflicts, particularly focusing on accusations against a major Western power and a major regional power. *   **Context:** The tweet operates within the context of Middle Eastern political discourse and conflict, specifically addressing conspiracy theories or alternative narratives regarding the formation and support of extremist or militant organizations. It references a statement attributed to a prominent American political figure concerning US intelligence activities and makes a separate assertion about another group's alleged affiliation with a specific Middle Eastern country known for its regional influence. This places it during a period where discussions about foreign intervention, the rise of extremist groups like the one mentioned by its common Arabic acronym, and regional power struggles (particularly involving the US and the named Middle Eastern country) were prominent. *   **Dialect:** The language used is formal and grammatically standard, employing vocabulary common across the Arab world for political discussion. There are no colloquialisms or regional markers. The dialect is **msa**. *   **Year:** The tweet likely dates from the period when the referenced extremist group was at its peak notoriety and territorial control, and when debates about its origins and the foreign policy decisions of the mentioned American politician (particularly related to her time as Secretary of State or subsequent campaigns) were widespread. This points towards approximately **2014-2017**. *   **Country:** Due to the use of MSA and the topic's broad regional relevance (US policy, Iranian influence, major extremist groups), it's difficult to pinpoint a single country. However, the specific nature of the accusation against the mentioned Middle Eastern nation is a common talking point in media and public discourse in several **Gulf countries** or potentially **Egypt**, where opposition to that nation's regional role is strong. It could theoretically originate from anywhere in the Arab world where MSA is used for online political commentary. *   **Sentiment Polarity:** The tweet expresses strong accusations and attributes nefarious actions (creation/backing of problematic groups) to powerful entities. This conveys disapproval and aligns with a critical or conspiratorial worldview regarding these entities. The sentiment is **negative**. *   **Impolite Words:** The tweet does not contain vulgarity or overtly rude language. **No**. *   **Hateful Words:** While strongly critical of national entities and a group widely considered extremist, it focuses on political actions/origins rather than attacking people based on intrinsic identity (like ethnicity or religion) using hateful tropes. It's accusatory political speech, not identity-based hate speech. **No**. *   **Contradictory Statements:** The tweet presents two distinct accusations about the origins/affiliations of different groups linked to different major powers. These claims are not inherently contradictory within the tweet's own logic, even if they represent complex or disputed geopolitical assertions. **No**. *   **Hidden Meanings:** Yes, the tweet implies a broader hidden reality where major global and regional powers are secretly manipulating events and creating destructive forces for their own ends. The underlying message is one of deep distrust towards official narratives and powerful states. **Yes**. *   **Double Entendre:** The language used is direct and political; there are no apparent plays on words with dual meanings. **No**. *   **Metaphors:** The tweet uses a term related to ""making"" or ""creation"" when referring to the groups' origins. While this could be seen as a very mild metaphor (groups aren't literally manufactured), it's used quite directly in this type of political discourse to mean engineered or deliberately brought about, rather than a figurative comparison to something else. It lacks strong metaphorical imagery. **No**. *   **Length:** The tweet consists of **10 Arabic words**. This is a **short** tweet. This length is effective because it delivers two impactful, controversial claims concisely, suitable for the rapid consumption format of Twitter. It assumes the audience is familiar with the underlying context and doesn't require lengthy explanation to understand the accusations. *   **Personality and Circumstances:** The author likely possesses a keen interest in regional and international politics, holds strong opinions, and exhibits significant distrust towards major powers (specifically the US and Iran in this context). They may subscribe to conspiracy theories or alternative news sources that frame geopolitical events as being driven by hidden manipulations. Their circumstances likely involve living in or being deeply concerned about the Middle East, feeling affected by the instability and conflicts, and attributing these issues to external interference. They might feel they are sharing important, non-mainstream information. The personality appears assertive and potentially cynical about official political narratives. *   **Motives:** The primary motives appear to be expressing a specific political viewpoint, assigning blame for regional problems, potentially influencing others who share similar distrust, and venting frustration or anger about the perceived manipulation of events by powerful entities. They might see themselves as revealing a hidden truth. *   **Relationship between elements:** The use of **MSA (dialect)** facilitates discussion of a broad **geopolitical topic (context)** across national borders, suitable for an audience potentially spanning the Arab world, perhaps particularly in the **Gulf or Egypt (country)** where the specific anti-Iran sentiment is common. The **year (~2014-2017)** aligns with the peak relevance of the groups and political figures mentioned in the **context**. The **personality** (distrustful, politically engaged) is drawn to such **topics/context** and uses **MSA** for wider reach. Their **circumstances** likely fuel their negative **sentiment** and motivate them to share these concise, impactful claims during that specific **time period**. The shortness of the tweet reflects the platform and the desire to make a quick, sharp point about this complex **context**.
Response must never contain any empty or new lines like \n or \r
'''

chat_non_sarcastic_analysis = model.start_chat(history=[])
chat_non_sarcastic_analysis.send_message(prompt)

print('chats ready!')


chats ready!


In [ ]:
def remove_arabic(text):
    text = re.sub(r'[\u0600-\u06FF]', '', text)
    return text

In [ ]:
def generate_analysis_batch(tweets, chat_analysis):
    try:
        prompt = '\n'.join(tweets)
        response = chat_analysis.send_message(prompt).text

        analysis = re.sub(r'\n\s*\n', '\n', response) # Remove empty lines to avoid wrongly zipping analysis with text
        analysis = analysis.split('\n')

        return analysis
    except Exception as e:
        print(f"Error in batch processing: {str(e)}")
        raise

In [ ]:
def generate_analysis(input_df, output_base, chat_analysis):
    ignored_indices = []
    total_rows = len(input_df)


    # Check if any batch files already exist
    existing_batches = [f for f in os.listdir(os.path.dirname(output_base))
                        if f.startswith(os.path.basename(output_base) + "_batch_")
                        and f.endswith(".csv")]

    # Determine the starting batch number
    start_batch = 0
    if existing_batches:
        batch_numbers = [int(f.split("_batch_")[1].split(".")[0]) for f in existing_batches]
        start_batch = max(batch_numbers) + 1
        start_index = start_batch * batch_size
        if start_index >= total_rows:
            print("All rows have been processed already!")
            return
        print(f"Continuing from batch {start_batch} (row {start_index}) out of {total_rows} rows")
    else:
        start_index = 0

    # Process rows in batches and save each to a separate file
    error_indices = []  # Store new error indices in this run
    try:
        # Calculate number of batches
        remaining_rows = total_rows - start_index
        num_batches = (remaining_rows + batch_size - 1) // batch_size
        pbar = tqdm(range(num_batches),
                    desc="Processing batches",
                    total=num_batches)

        for batch_num in pbar:
            time.sleep(2)

            current_batch = start_batch + batch_num
            batch_start = start_index + batch_num * batch_size
            batch_end = min(batch_start + batch_size, total_rows)

            # Define batch output file
            batch_output_file = f"{output_base}_batch_{current_batch}.csv"

            batch_rows = []
            batch_texts = []

            # Prepare batch data
            for index in range(batch_start, batch_end):
                if index in ignored_indices:
                    continue

                try:
                    row = input_df.iloc[index]
                    batch_rows.append(row)
                    batch_texts.append(row['text'])

                except Exception as e:
                    error_msg = f"Error preparing index {index}: {str(e)}"
                    pbar.write(error_msg)
                    error_indices.append(index)
                    continue

            try:
                # Process the batch
                pbar.set_description(f"Processing batch {current_batch} ({len(batch_texts)} texts)")
                analyses = generate_analysis_batch(
                    batch_texts, copy.deepcopy(chat_analysis)
                )

                # Create output rows
                batch_results = []
                for i, (row, analysis) in enumerate(zip(batch_rows, analyses)):
                    batch_results.append({
                        'text': row['text'],
                        'sarcasm': row['sarcasm'],
                        'sentiment': row['sentiment'],
                        'dialect': row['dialect'],
                        'combined_label': row['combined_label'],
                        'analysis': re.sub(r'^[^:]*:', '', analysis),
                        'correct_analysis_tweet': analysis.split(':')[0] == row['text']
                    })

                # Write batch results to a separate CSV file
                batch_df = pd.DataFrame(batch_results)
                batch_df.to_csv(batch_output_file, index=False)
                pbar.write(f"Saved batch {current_batch} to {batch_output_file}")

            except Exception as e:
                error_msg = f"Error processing batch {current_batch}: {str(e)}"
                pbar.write(error_msg)
                # Add all indices in this failed batch to error list
                for i, index in enumerate(range(batch_start, batch_end)):
                    if index not in ignored_indices:
                        error_indices.append(index)

            if len(error_indices) >= 3:
                pbar.write("Reached 3 errors, stopping process")
                break

    except KeyboardInterrupt:
        print("\nProcess interrupted by user")
    finally:
        # Print summary
        print("\nProcessing summary:")
        last_processed = start_index + batch_num * batch_size
        if 'batch_end' in locals():
            last_processed = batch_end - 1
        print(f"Last processed index: {min(last_processed, total_rows-1)}")
        if error_indices:
            print("\nNew error indices in this run:")
            print(error_indices)
            print("\nTo ignore these indices in future runs, add them to ignored_indices list:")
            print(f"ignored_indices = {ignored_indices + error_indices}")

    print("Processing completed!")


In [ ]:
def combine_batch_files(input_path, output_file):
    batch_dfs = []

    for filename in os.listdir(input_path):
        if '_batch_' in filename and filename.endswith('.csv'):
            file_path = os.path.join(input_path, filename)
            df = pd.read_csv(file_path)
            batch_dfs.append(df)

    if batch_dfs:
        combined_df = pd.concat(batch_dfs, ignore_index=True)
        combined_df.to_csv(f'{input_path}/{output_file}', index=False)
        print(f"Combined {len(batch_dfs)} batch files into {output_file}")
    else:
        print("No batch CSV files found")

In [ ]:
def check_and_fix_incorrect_batches(input_path, chat_analysis):
  # First collect all incorrect analyses
  incorrect_analyses = []

  # Gather incorrect analyses from all batches
  for filename in os.listdir(input_path):
      if '_batch_' in filename and filename.endswith('.csv'):
          file_path = os.path.join(input_path, filename)
          df = pd.read_csv(file_path)

          # Get incorrect rows
          incorrect_mask = df['correct_analysis_tweet'] == False
          if incorrect_mask.any():
              incorrect_batch = df[incorrect_mask].copy()
              incorrect_batch['original_batch'] = filename
              incorrect_analyses.append(incorrect_batch)

  if not incorrect_analyses:
      print("All analyses are correct!")
      return

  # Combine all incorrect analyses into one DataFrame
  all_incorrect_df = pd.concat(incorrect_analyses, ignore_index=True)
  incorrect_file = os.path.join(input_path, 'incorrect_analyses.csv')
  all_incorrect_df.to_csv(incorrect_file, index=False)
  print(f"Found {len(all_incorrect_df)} incorrect analyses")

  # Process incorrect analyses in batches
  for i in range(0, len(all_incorrect_df), batch_size):
      batch = all_incorrect_df.iloc[i:i+batch_size]
      batch_texts = batch['text'].tolist()

      try:
          # Regenerate analyses for batch
          new_analyses = generate_analysis_batch(batch_texts, copy.deepcopy(chat_analysis))

          # Update original batch files with corrections
          for _, row in batch.iterrows():
              original_batch = row['original_batch']
              original_file = os.path.join(input_path, original_batch)

              # Read original batch file
              batch_df = pd.read_csv(original_file)

              # Find and update the specific row
              mask = batch_df['text'] == row['text']
              if mask.any():
                  idx = mask.idxmax()
                  analysis = new_analyses[batch_texts.index(row['text'])]

                  batch_df.loc[idx, 'analysis'] = re.sub(r'^[^:]*:', '', analysis)
                  batch_df.loc[idx, 'correct_analysis_tweet'] = analysis.split(':')[0] == row['text']

              # Save updated batch file
              batch_df.to_csv(original_file, index=False)

      except Exception as e:
          print(f"Error fixing batch starting at index {i}: {str(e)}")
          continue


In [ ]:
!mkdir /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic
!mkdir /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis
!mkdir /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic
!mkdir /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis
!echo 'Created!'

Created!


In [ ]:
output_base = '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/'
input_df = training_df[training_df['sarcasm'] == True]

chat_analysis = chat_sarcastic_analysis

generate_analysis(input_df, output_base, chat_analysis)
check_and_fix_incorrect_batches(output_base, chat_analysis)
combine_batch_files(output_base, "generated_analysis_sarcastic.csv")

print('Done :)')

Continuing from batch 16 (row 480) out of 1530 rows


Processing batch 27 (30 texts):  17%|█▋        | 6/35 [00:00<00:00, 53.59it/s]

Saved batch 16 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_16.csv
Saved batch 17 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_17.csv
Saved batch 18 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_18.csv
Saved batch 19 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_19.csv
Saved batch 20 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_20.csv
Saved batch 21 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_21.csv
Saved batch 22 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_22.csv
Saved batch 23 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_23.csv
Saved batch 24 to /content/drive/MyDrive

Processing batch 37 (30 texts):  51%|█████▏    | 18/35 [00:00<00:00, 50.18it/s]

Saved batch 27 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_27.csv
Saved batch 28 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_28.csv
Saved batch 29 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_29.csv
Saved batch 30 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_30.csv
Saved batch 31 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_31.csv
Saved batch 32 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_32.csv
Saved batch 33 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_33.csv
Saved batch 34 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_34.csv
Saved batch 35 to /content/drive/MyDrive

Processing batch 48 (30 texts):  86%|████████▌ | 30/35 [00:00<00:00, 50.12it/s]

Saved batch 37 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_37.csv
Saved batch 38 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_38.csv
Saved batch 39 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_39.csv
Saved batch 40 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_40.csv
Saved batch 41 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_41.csv
Saved batch 42 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_42.csv
Saved batch 43 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_43.csv
Saved batch 44 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_44.csv
Saved batch 45 to /content/drive/MyDrive

Processing batch 50 (30 texts): 100%|██████████| 35/35 [00:00<00:00, 51.32it/s]


Saved batch 48 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_48.csv
Saved batch 49 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_49.csv
Saved batch 50 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/_batch_50.csv

Processing summary:
Last processed index: 1529
Processing completed!
All analyses are correct!
Combined 51 batch files into generated_analysis_sarcastic.csv
Done :)


In [ ]:
output_base = '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/'
input_df = training_df[training_df['sarcasm'] == False]

chat_analysis = chat_non_sarcastic_analysis

generate_analysis(input_df, output_base, chat_analysis)
check_and_fix_incorrect_batches(output_base, chat_analysis)
combine_batch_files(output_base, "generated_analysis_non_sarcastic.csv")

print('Done :)')

Continuing from batch 115 (row 3450) out of 8623 rows


Processing batch 122 (30 texts):   3%|▎         | 5/173 [00:00<00:03, 43.73it/s]

Saved batch 115 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_115.csv
Saved batch 116 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_116.csv
Saved batch 117 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_117.csv
Saved batch 118 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_118.csv
Saved batch 119 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_119.csv
Saved batch 120 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_120.csv
Saved batch 121 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_121.csv


Processing batch 123 (30 texts):   3%|▎         | 5/173 [00:00<00:03, 43.73it/s]

Saved batch 122 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_122.csv


...

Saved batch 277 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_277.csv
Saved batch 278 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_278.csv
Saved batch 279 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_279.csv


Processing batch 286 (30 texts):  98%|█████████▊| 170/173 [00:04<00:00, 41.82it/s]

Saved batch 280 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_280.csv
Saved batch 281 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_281.csv
Saved batch 282 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_282.csv
Saved batch 283 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_283.csv
Saved batch 284 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_284.csv
Saved batch 285 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_285.csv


Processing batch 287 (13 texts): 100%|██████████| 173/173 [00:04<00:00, 40.34it/s]


Saved batch 286 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_286.csv
Saved batch 287 to /content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/_batch_287.csv

Processing summary:
Last processed index: 8622
Processing completed!
All analyses are correct!
Combined 288 batch files into generated_analysis_non_sarcastic.csv
Done :)


In [ ]:
analysis_1 = pd.read_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/analysis/generated_analysis_sarcastic.csv')
analysis_2 = pd.read_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/analysis/generated_analysis_non_sarcastic.csv')

analysis = pd.concat([analysis_1, analysis_2], ignore_index=True)

analysis.drop(columns=['correct_analysis_tweet'], inplace=True)
analysis['analysis'] = analysis['analysis'].apply(remove_arabic)

analysis.to_csv('/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/analysis.csv', index=False)

In [ ]:
!rm -r '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/sarcastic/'
!rm -r '/content/drive/MyDrive/Project/ArSarcasm-v2/02_augment_gemini_prompt/non_sarcastic/'

!echo 'Temp directories deleted!'

Temp directories deleted!


In [ ]:
import time
def shutdown_sys():
  print('Will Shutdown')
  time.sleep(30)
  from google.colab import runtime
  runtime.unassign()

In [ ]:
shutdown_sys()

Will Shutdown
